# 04 Ranking-aware experiment (Experiment B)

**Question.** Does a pairwise ranking term change ordering quality relative to A?

**Type.** Pair diagnostics (labels only) plus run/inspection ([spec](../docs/experiments/b-ranking-aware.md)). Config values are untuned placeholders until the declared grid is run on validation data.

In [ ]:
import json, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
import os; os.chdir(REPO)
import numpy as np
import torch
from ct_iqa.data.sampling import ordered_pair_mask
labels = json.loads(Path("data/labels/ldct_iqac/train.json").read_text())
manifest = json.loads(Path("experiments/study_v2/protocol/split_manifest.json").read_text())
rng = np.random.default_rng(0)
batch = torch.tensor([labels[n] for n in rng.choice(manifest["train"], 64, replace=False)])
for delta in (0.0, 0.25, 0.5, 1.0):
    m = ordered_pair_mask(batch, delta)
    print(f"min_score_diff={delta}: valid pairs {int(m.sum())} of {64 * 63 // 2}")

In [ ]:
import json, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
import os; os.chdir(REPO)
import numpy as np
from ct_iqa.config.loader import load_study_config
from ct_iqa.experiment.runs import latest_completed_run, read_predictions
CONFIG = "configs/experiments/b_ranking_aware.yaml"
cfg = load_study_config(CONFIG)
print(cfg.run.experiment_id, cfg.model.name, cfg.loss.name, "bn_eps =", cfg.model.bn_eps)

## Run (explicit, expensive)
Set `RUN_TRAINING = True` only with an approved compute budget; this trains for the configured number of epochs and writes a new immutable run directory.

In [ ]:
RUN_TRAINING = False
DEVICE = "cpu"   # use "cuda" when available
if RUN_TRAINING:
    import dataclasses
    from ct_iqa.experiment.run import run_experiment
    cfg = dataclasses.replace(cfg, train=dataclasses.replace(cfg.train, device=DEVICE))
    run_dir = run_experiment(cfg, config_path=CONFIG)
else:
    run_dir = latest_completed_run(cfg.run.experiment_id)
print("run directory:", run_dir if run_dir else "NO COMPLETED RUN FOUND (status PENDING)")

In [ ]:
if run_dir:
    print(json.loads((run_dir / "metrics_validation.json").read_text()))
    hist = json.loads((run_dir / "history.json").read_text())
    print("selected epoch (1-based):", hist["best_epoch"], "policy:", hist["selection_policy"])

In [ ]:
if run_dir:
    parts = hist["epochs"][-1]["train_loss_parts"]
    print("last-epoch mean per-batch diagnostics:", parts)